# Analysis Pipeline: Accessibility and Usability of Complex Computer Systems

This notebook executes the full research pipeline:
1. **Data Loading & Cleaning**: Loads raw session data, filters incomplete sessions, and imputes missing SUS scores.
2. **Statistical Analysis**: Performs Repeated Measures ANOVA with Holm-Bonferroni correction.
3. **Power Analysis**: Computes statistical power and flags underpowered subgroups.
4. **Visualization**: Generates publication-quality box plots.
5. **Reporting**: Compiles results into summary artifacts.

**Constraint**: This pipeline fails if `data/raw/` is empty unless `--simulate` (dev mode) is used. For this notebook, we assume the cleaning and analysis modules have produced the necessary intermediate files. If raw data is missing, the pipeline will attempt to load from `data/processed/cleaned_sessions.csv`.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Check for raw data existence (NFR-002 enforcement)
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_sessions.csv"

if not RAW_DATA_DIR.exists() or not list(RAW_DATA_DIR.glob("*.json")):
    if not CLEAN_DATA_PATH.exists():
        print("⚠️ WARNING: No raw data found in data/raw/ and no cleaned data found.")
        print("This pipeline requires real data. If running in dev mode, ensure data is generated or available.")
        print("Attempting to proceed with existing cleaned data if available, else failing.")
    else:
        print("ℹ️ Raw data missing, but cleaned data found at data/processed/cleaned_sessions.csv. Proceeding with analysis.")
else:
    print("ℹ️ Raw data found. Cleaning and analysis will proceed.")

## Step 1: Data Cleaning Pipeline

We invoke the `clean_data` module to ensure data is ready for analysis. This step handles:
- Exclusion of incomplete sessions (`status='incomplete'`)
- SUS score imputation (mean imputation for ≤1 missing item)
- Type coercion and validation

In [ ]:
from analysis.clean_data import main as clean_main

clean_input = str(RAW_DATA_DIR)
clean_output = str(CLEAN_DATA_PATH)

# Check if cleaning is needed
if not CLEAN_DATA_PATH.exists():
    print("Running data cleaning pipeline...")
    # Simulate CLI arguments for clean_data
    sys.argv = ['clean_data', '--input', clean_input, '--output', clean_output]
    try:
        clean_main()
        print("✅ Data cleaning completed successfully.")
    except SystemExit as e:
        if e.code != 0:
            raise RuntimeError("Data cleaning pipeline failed. Ensure raw data exists and is valid.")
else:
    print("✅ Cleaned data already exists at data/processed/cleaned_sessions.csv. Skipping cleaning step.")

## Step 2: Statistical Analysis Pipeline

This step performs:
- Shapiro-Wilk normality test (audit only)
- Repeated Measures ANOVA (mandated by FR-002)
- Holm-Bonferroni correction for multiple comparisons
- Effect size calculation

In [ ]:
from analysis.run_analysis import run_analysis_pipeline

analysis_input = str(CLEAN_DATA_PATH)
analysis_output_dir = PROJECT_ROOT / "data" / "processed"

print("Running statistical analysis pipeline...")
try:
    run_analysis_pipeline(
        input_path=analysis_input,
        output_dir=analysis_output_dir
    )
    print("✅ Statistical analysis completed successfully.")
    print(f"Output files written to: {analysis_output_dir}")
except Exception as e:
    print(f"❌ Statistical analysis failed: {e}")
    raise

## Step 3: Power Analysis

Compute statistical power given N, effect size (eta-squared), and α = 0.05.
Flag subgroups as 'UNDERPOWERED' if N < 30.

In [ ]:
from analysis.power_analysis import PowerCalculator

metrics_summary_path = analysis_output_dir / "metrics_summary.csv"
power_output_path = analysis_output_dir / "power_flags.json"

if metrics_summary_path.exists():
    df_metrics = pd.read_csv(metrics_summary_path)
    # Load cleaned data to get N per subgroup
    df_clean = pd.read_csv(CLEAN_DATA_PATH)
    
    calculator = PowerCalculator()
    results = calculator.compute_power(df_clean, df_metrics)
    
    with open(power_output_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"✅ Power analysis completed. Results saved to {power_output_path}")
    print(json.dumps(results, indent=2))
else:
    print("⚠️ metrics_summary.csv not found. Skipping power analysis.")

## Step 4: Visualization

Generate publication-quality box plots for:
- Completion Time
- Error Count
- SUS Score

In [ ]:
from analysis.visualizer import Visualizer

figures_dir = PROJECT_ROOT / "figures"
figures_dir.mkdir(exist_ok=True)

if CLEAN_DATA_PATH.exists():
    df = pd.read_csv(CLEAN_DATA_PATH)
    
    visualizer = Visualizer()
    
    # Completion Time
    visualizer.plot_completion_time(df, output_path=str(figures_dir / "completion_time.png"))
    
    # Error Count
    visualizer.plot_error_count(df, output_path=str(figures_dir / "error_count.png"))
    
    # SUS Score
    visualizer.plot_sus_score(df, output_path=str(figures_dir / "sus_score.png"))
    
    print("✅ Visualizations generated successfully.")
    print(f"Output files: {list(figures_dir.glob('*.png'))}")
else:
    print("⚠️ Cleaned data not found. Skipping visualization.")

## Step 5: Report Generation

Compile all results into a final report.

In [ ]:
from analysis.report_generator import ReportGenerator

report_output_path = analysis_output_dir / "report_summary.txt"

if CLEAN_DATA_PATH.exists() and metrics_summary_path.exists():
    report_gen = ReportGenerator()
    report_gen.generate_report(
        cleaned_data_path=str(CLEAN_DATA_PATH),
        metrics_summary_path=str(metrics_summary_path),
        power_flags_path=str(power_output_path),
        output_path=str(report_output_path)
    )
    print(f"✅ Report generated at {report_output_path}")
else:
    print("⚠️ Required data files not found. Skipping report generation.")

## Final Verification

Verify that all required artifacts have been produced.

In [ ]:
required_artifacts = [
    "data/processed/cleaned_sessions.csv",
    "data/processed/metrics_summary.csv",
    "data/processed/power_flags.json",
    "data/processed/normality_log.txt",
    "data/processed/report_summary.txt",
    "figures/completion_time.png",
    "figures/error_count.png",
    "figures/sus_score.png"
]

all_present = True
for artifact in required_artifacts:
    path = PROJECT_ROOT / artifact
    if path.exists():
        print(f"✅ {artifact}")
    else:
        print(f"❌ {artifact} MISSING")
        all_present = False

if all_present:
    print("\n🎉 All required artifacts have been generated successfully.")
else:
    print("\n⚠️ Some artifacts are missing. Check the pipeline logs for errors.")